# M3L4 E04 — Golden Dataset para routing
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

---

## Qué necesitas saber antes

| Módulo | Concepto | Por qué lo necesitas acá |
|---|---|---|
| M3L4 E02 | `route_query()`, sistema multi-agente | El router que evaluamos es el mismo de E02 |
| M3L4 E03 | Misclassification como patrón de falla | El golden dataset mide exactamente eso |
| M3L1 | Tool contracts, testing de tools | El dataset es un contrato de testing |
| Python | `pandas.DataFrame`, `groupby`, `mean` | Para analizar resultados de la evaluación |

Si no viste E02, el router `route_query()` clasifica consultas en intents (hr, it, finance, legal, etc.).

---

## Definiciones clave

| Concepto | Definición simple | Cómo aparece en este notebook |
|---|---|---|
| **Golden dataset** | Conjunto fijo de casos de prueba con respuesta esperada conocida | Lista de dicts con `id`, `query`, `expected_intent` |
| **Routing accuracy** | Proporción de consultas que el router envía al intent correcto | `accuracy = df['correct'].mean()` |
| **Ground truth** | Valor esperado o verdadero contra el que comparamos | `expected_intent` en cada caso del dataset |
| **Caso incorrecto** | Consulta donde `actual_intent != expected_intent` | Fila en `df` con `correct == 0` |
| **Accuracy por intent** | Precisión del router separada por cada categoría | `df.groupby('expected_intent')['correct'].mean()` |
| **Patrón de fallo** | Tipo de consulta donde el router se equivoca sistemáticamente | Casos `hr` clasificados como `general`, etc. |

---

## Cómo encaja esto en un sistema de agentes

```
E02: Sistema multi-agente con tracing
    |  route_query() decide el intent
    v
E03: Diagnosticar fallas en traces individuales
    |  Por cada request, vemos si hubo misclassification
    v
E04: Golden dataset para medir el router (ESTE EJERCICIO)
    |  Prevención: medir sistemáticamente antes de producir fallas
    v
E05: Router v1 vs v2: comparar accuracy entre versiones
    |  Mejora continua del routing
```

**Objetivo del ejercicio:** usar un golden dataset para medir si el orquestador envía cada consulta al agente correcto.

## Instalación e imports

| Import | Qué hace | Por qué lo necesitamos |
|---|---|---|
| `pandas` (via pip) | DataFrames para análisis tabular de resultados | Para mostrar resultados, calcular accuracy y agrupar por intent |
| `pd.DataFrame` | Estructura de datos con filas y columnas | Almacenar id, query, expected, actual, correct de cada caso |
| `.mean()` | Calcula el promedio de una columna | `df['correct'].mean()` = proporción de aciertos |
| `.groupby()` | Agrupa datos por una columna categórica | `df.groupby('expected_intent')['correct'].mean()` = accuracy por intent |

```python
!pip install pandas -q
import pandas as pd
```

In [ ]:
!pip install pandas -q
import pandas as pd
print('pandas listo.')

## Router de referencia (ya dado)

Este es el mismo `route_query()` de E02. Clasifica consultas en 6 intents basándose en palabras clave.

### Reglas del router

| Palabras clave | Intent |
|---|---|
| vacaciones, licencia, recibo, portal rrhh | `hr` |
| vpn, laptop, error, app, wifi, login | `it` |
| factura, pago, reembolso | `finance` |
| contrato, legal, confidencialidad, nda | `legal` |
| Menos de 3 palabras | `clarification` |
| Ninguna de las anteriores | `general` |

In [ ]:
def route_query(query: str) -> str:
    q = query.lower()
    if any(w in q for w in ['vacaciones', 'licencia', 'recibo', 'portal rrhh']):
        return 'hr'
    if any(w in q for w in ['vpn', 'laptop', 'error', 'app', 'wifi', 'login']):
        return 'it'
    if any(w in q for w in ['factura', 'pago', 'reembolso']):
        return 'finance'
    if any(w in q for w in ['contrato', 'legal', 'confidencialidad', 'nda']):
        return 'legal'
    if len(q.split()) <= 2:
        return 'clarification'
    return 'general'

print('Router listo.')

## Golden Dataset

Conjunto de 8 casos de prueba que cubren todos los intents. Cada caso tiene:

- `id`: identificador único
- `query`: texto de la consulta
- `expected_intent`: el intent correcto según un experto humano

> **Frase clave:** el golden dataset permite pasar de "creo que funciona" a "mejoró de X% a Y%".

```python
golden_dataset = [
    {'id': 'case_001', 'query': '...', 'expected_intent': 'hr'},
    ...
]
```

In [ ]:
golden_dataset = [
    {'id': 'case_001', 'query': '¿Cómo solicito mis días de vacaciones?',          'expected_intent': 'hr'},
    {'id': 'case_002', 'query': 'Mi VPN no conecta desde ayer',                     'expected_intent': 'it'},
    {'id': 'case_003', 'query': 'Necesito ver mi factura del mes pasado',            'expected_intent': 'finance'},
    {'id': 'case_004', 'query': 'Necesito el contrato de confidencialidad actualizado', 'expected_intent': 'legal'},
    {'id': 'case_005', 'query': 'No puedo entrar al portal para ver mi recibo',     'expected_intent': 'hr'},
    {'id': 'case_006', 'query': 'ayuda',                                             'expected_intent': 'clarification'},
    {'id': 'case_007', 'query': '¿Cuándo se procesa el reembolso de gastos?',       'expected_intent': 'finance'},
    {'id': 'case_008', 'query': 'El sistema de login no me deja entrar',             'expected_intent': 'it'},
]

print(f'Golden dataset cargado: {len(golden_dataset)} casos')

## TODO — Evaluar el router

Completa la función `evaluate_router` que:

1. Recorre el dataset caso por caso
2. Aplica el router a cada query
3. Compara `actual_intent` con `expected_intent`
4. Retorna un DataFrame con los resultados y el accuracy global

### Desglose de `evaluate_router(router_fn, dataset)`

| Parámetro | Tipo | Qué es |
|---|---|---|
| `router_fn` | `function` | La función de routing (ej: `route_query`) |
| `dataset` | `list[dict]` | Lista de casos con `query` y `expected_intent` |
| **Retorna** | `tuple[DataFrame, float]` | (df con resultados, accuracy entre 0 y 1) |

### Columnas del DataFrame resultado

| Columna | Descripción |
|---|---|
| `id` | Identificador del caso |
| `query` | Consulta original |
| `expected_intent` | Intent esperado (ground truth) |
| `actual_intent` | Intent que devolvió el router |
| `correct` | 1 si coincide, 0 si no |

In [ ]:
def evaluate_router(router_fn, dataset: list) -> tuple:
    """
    Evalúa un router sobre un dataset.

    Returns:
        df: DataFrame con columnas [id, query, expected_intent, actual_intent, correct]
        accuracy: float entre 0 y 1
    """
    rows = []
    for case in dataset:
        # TODO: aplicar router_fn al query
        actual_intent = None  # reemplazar

        # TODO: calcular si es correcto (1 o 0)
        correct = None  # reemplazar

        rows.append({
            'id': case['id'],
            'query': case['query'],
            'expected_intent': case['expected_intent'],
            'actual_intent': actual_intent,
            'correct': correct
        })

    df = pd.DataFrame(rows)
    # TODO: calcular accuracy (media de la columna 'correct')
    accuracy = None  # reemplazar

    return df, accuracy

print('Función definida.')

In [ ]:
df, accuracy = evaluate_router(route_query, golden_dataset)
print(f'Routing accuracy: {accuracy:.2%}')
df

## TODO — Análisis de fallos

Identifica los casos incorrectos y analiza el patrón de fallos.

### Preguntas para responder

1. ¿Qué casos fallaron? ¿Por qué?
2. ¿Hay un patrón en las fallas? (ej: todos los fallos son de HR)
3. ¿Qué palabras clave agregarías al router para corregirlo?

### Por qué importa el análisis por intent

```python
accuracy_by_intent = df.groupby('expected_intent')['correct'].mean()
```

Esto muestra el accuracy separado por cada categoría. Un accuracy global del 80% puede esconder que un intent particular tiene 0% de acierto.

In [ ]:
# TODO: filtrar el DataFrame para mostrar solo los casos incorrectos
df_failures = None  # reemplazar
df_failures

In [ ]:
# TODO: calcular accuracy por intent usando groupby
accuracy_by_intent = None  # reemplazar
accuracy_by_intent

In [ ]:
assert df is not None, 'df es None'
assert accuracy is not None, 'accuracy es None'
assert 0 <= accuracy <= 1, 'accuracy fuera de rango'
assert 'correct' in df.columns, 'Falta columna correct'
assert 'actual_intent' in df.columns, 'Falta columna actual_intent'
print(f'Checks E04 OK — Routing accuracy: {accuracy:.2%}')

## Errores comunes

| Error | Causa | Cómo detectarlo |
|---|---|---|
| No llamar `router_fn(query)` | `actual_intent` queda como `None` | Todas las filas tienen `actual_intent = None` |
| Comparar mal intent | Usar `!=` en vez de `==` para `correct` | Todos los casos aparecen como incorrectos |
| No calcular `.mean()` | `accuracy` queda como suma en vez de proporción | Accuracy > 1 (si sumaste en vez de promediar) |
| Olvidar `groupby` para accuracy por intent | Solo mirar accuracy global | No se detecta que un intent particular falla siempre |
| Dataset incompleto | No cubrir todos los intents posibles | Algunos intents no están evaluados |

## Síntesis

### Qué construiste

| Componente | Descripción |
|---|---|
| `evaluate_router()` | Función que mide accuracy de cualquier router contra un golden dataset |
| DataFrame de resultados | Vista tabular de cada caso con su acierto/fallo |
| Accuracy global | Métricas resumen: qué % de casos clasifica bien el router |
| Accuracy por intent | Precisión separada por categoría (HR, IT, Finance, Legal...) |
| Análisis de fallos | Identificación de patrones: dónde se equivoca el router |

### Flujo completo de evaluación

```
Golden dataset  -->  Router  -->  Comparación  -->  Accuracy  -->  Mejora
(casos con        (actual        expected vs       (métricas       (iterar:
 ground truth)     intent)        actual)           cuantitativas)  agregar casos,
                                                                   ajustar reglas)
```

### Relación con otros ejercicios

| Ejercicio | Conexión con E04 |
|---|---|
| **E05** | Router v1 vs v2: comparar accuracy entre dos versiones usando el mismo dataset |
| **E06** | Evaluator agent: automatizar la comparación con un LLM como juez |
| **E11** | Golden dataset scores: medir mejora continua en el tiempo |